In [1]:
### Quick intros to python topics

In [2]:
# Shapely .bounds: (minx, miny, maxx, maxy)
# QGraphicsItem.grabMouse() # set mouse grabber item


In [ ]:
from utils import * 
textItem = QGraphicsSimpleTextItem('')
print(textItem.isVisible())
textItem.setText('hey')
print(textItem.text())
print(textItem.brush())

In [ ]:
# Saura testing
from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
from View import View
from BoardScene import BoardScene
from utils import Utils
from collections import defaultdict
import sys
from utils import LayerItem
from CopperItemContainer import CopperItemContainer
from MainWindow import MainWindow
from Via import Via

from NetSymbol import NetSymbol
from Trace import Trace

    
window = MainWindow()

scene = window.centralWidget().widget(1).scene()
window.show()

e = QGraphicsEllipseItem(-10,-10,20,20) 
e2 = QGraphicsEllipseItem(-10,-10,20,20, e) 
e3 = QGraphicsEllipseItem(-10,-10,20,20 , e2)
print('CHILDITEMS:', e.childItems())

sys.exit(qApp.exec())

# e = QGraphicsEllipseItem(-10,-10,20,20)
# e.setFlags(QGraphicsItem.ItemIsMovable | QGraphicsItem.ItemIsSelectable)
# t = QGraphicsSimpleTextItem('' , e)
# t.setFont(Utils.symbolFont)
# t.setText('hey')
# scene.addItem(e)

# via = Via(50, 30, layers = ['F.Cu', 'B.Cu'])
# via.setNet('GND')

# via3V3 = Via(50, 30, layers = ['F.Cu', 'B.Cu'])
# via3V3.setNet('3V3')


# trace = Trace(50,50 , 1000, 800, layers = ['F.Cu'], traceWidth = 1)
# trace.setLine(QLineF(0,0, 100,100))

# # print('TRACE.LINE():', trace.line())
# scene.addItem(trace)

# scene.addItem(via)
# scene.addItem(via3V3)
# via.setPos(-100,-100)


# # print([item for item in via.childItems()])
# ns = NetSymbol('?', 1, "symbols/NetSymbols/GND.sym")

ADDING ITEM OF TYPE: <class 'PySide6.QtWidgets.QGraphicsEllipseItem'>
NONLAYERSITEM <PySide6.QtWidgets.QGraphicsEllipseItem(0x1f0508fe250, pos=0,0) at 0x000001F0112CC740>  ADDED TO BOARDSCENE
ADDING ITEM OF TYPE: <class 'LayersItem.LayersEllipseItem'>

SET MODE TO BoardSceneMode.NormalMode
CHILDITEMS: [<PySide6.QtWidgets.QGraphicsEllipseItem(0x1f0122fc8a0, parent=0x1f0122fc9c0, pos=0,0) at 0x000001F00C5AC7C0>]
MYTABLEWIDGET.MOUSEPRESSEVENT
PRESSED THIS RECORD: {'primary_attributes': 'capacitance,voltage_rated,package/case', 'symbol': 'C:/Users/robby/OneDrive/Saura/symbols/GRM21BR61E106KA73L.sym', 'footprint': 'C:/Users/robby/OneDrive/Saura/footprints/SMD/G-21_MUR.fp', 'package/case': '0805 (2012 Metric)', 'datasheet': 'https://search.murata.co.jp/Ceramy/image/img/A01X/G101/ENG/GRM21BR61E106KA73-01.pdf', 'reference': 'C', 'mpn': 'GRM21BR61E106KA73L', 'unit_price': '0.24', 'vendor_part_page': 'https://www.digikey.com/en/products/detail/murata-electronics/GRM21BR61E106KA73L/2334874', 'mfr

In [ ]:
# Qt Quirk: This one cost me about 20 hours # https://doc.qt.io/qt-6/qpainter.html

# QPainter has many overloads for .drawLine. Some are float, some are int. 
# "All of these (QPainter convenience) functions have both integer and floating point versions."
# HOWEVER. The overload which takes four numbers ONLY ACCEPTS INTEGER, there is no float version. So if you want to use floats, you better NOT use that overload, and pass intead two QPointFs, or a QLineF.
# Here are all the overloads: 
# void 	drawLine(const QLineF &line)
# void 	drawLine(const QLine &line)
# void 	drawLine(const QPoint &p1, const QPoint &p2)
# void 	drawLine(const QPointF &p1, const QPointF &p2)
# void 	drawLine(int x1, int y1, int x2, int y2) # NOte this one has no float equivalent 



In [ ]:
class A: 
    @staticmethod 
    def func(a): 
        print(a) 

    def func2(self):
        self.func('a')

a = A() 
a.func2()

a


In [ ]:
from utils import * 
print(QPoint(10,10) + QPoint(10,40))
x = QPointF(10.1 , 10.2) 
print(x.toPoint())
print(x)

PySide6.QtCore.QPoint(20, 50)
PySide6.QtCore.QPoint(10, 10)
PySide6.QtCore.QPointF(10.100000, 10.200000)


In [ ]:
# I could not get Traces to draw from seeker center in BoardScene-- Try from scratch here
from utils import * 
from View import View 
from Ffline import Ffline
from Trace import Trace 

class PracticeScene(QGraphicsScene ): 
    dpi = qApp.screens()[0].physicalDotsPerInch()
    dpmm = dpi / 25.4
    # * 1 bc IDK what the boardScene grid step should be, and * 1 makes for a 1mm grid step.
    grid_spacing_pixels=  dpi * Utils.gridPt1mm # the spacing at which to snap to. 
    gridSpacingMm = 1 # as in 1mm

    def __init__(self):
        super().__init__()
        self._activeLayer = 'F.Cu' 
        self._traceWidth = 1.0
        
        self.ffline = None 
        
        # self.views()[0].setMouseTracking(True)
        self.seeker = QGraphicsEllipseItem(-5,-5,10,10)
        self.seeker.setPen(QPen(Qt.green , 0))
        self.addItem(self.seeker)

        testTrace = Trace(0,0,100,100 , ['B.Cu'], traceWidth=1)
        # testTrace = Trace(50,50 , 1000, 800, layers = ['F.Cu'], traceWidth = 1)
        testTrace.setLine(QLineF(0,0, 100,100))
        # testTrace = Trace.fromLine(QLineF(0,0,110,100) , 1 , ['B.Cu'])
        testTrace = Trace.fromPoints(['B.Cu'] , 1 , QPointF(0,0) , QPointF(100.5 , 100.5))
        self.addItem(testTrace)

        e1 = QGraphicsEllipseItem(-1,-1,2,2) 
        e1.setPen(QPen(Qt.black , 0))
        e1.setPos(100,100)
        self.addItem(e1)
        
    def snapToGrid(self, point: QPointF ):
        gridSpacing = self.gridSpacingMm * self.dpi/25.4 # Conversion of Xmm to pixels
        return Utils.snapToGrid(point, gridSpacing)

    def mousePressEvent(self, event):
            
        self.startPosition = self.snapToGrid(event.scenePos()) 

        self.ffline =Ffline(self.startPosition , self.startPosition , self)
        self.ffline.setPoints(self.startPosition , self.snapToGrid(event.scenePos()) )
        
    def mouseMoveEvent(self, event):
        self.seeker.setPos(self.snapToGrid(event.scenePos()))
        if self.ffline: 
            self.ffline.setPoints(self.startPosition , self.seeker.scenePos())
        
    def activeLayer(self): 
        return self._activeLayer
    def setActiveLayer(self, activeLayer):
        self._activeLayer = activeLayer

    def traceWidth(self):
        return self._traceWidth
    def setTraceWidth(self,traceWidth):
        self._traceWidth = float(traceWidth)


view = View() 
scene = PracticeScene()
view.setScene(scene)

view.show() 

sys.exit(app.exec())



SETTRACEWIDTH DONE
SET SCENE BOUNDS:  (-0.5488155364580245, -0.5488155364580245, 100.54881553645802, 100.54881553645802)
SETTRACEWIDTH DONE


SystemExit: 0

c:\Users\robby\OneDrive\Saura\myenv\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
class A():
    def a(self):
        return self._a 

    @staticmethod
    def a():
        return 'aa'

a = A() 
print(a.a())
print(A.a())
# static method shadows instance methods. Cant have a static method and instance method of same name.

In [ ]:
print(abs(1.3))

In [ ]:
print(int(3.9))

In [ ]:
from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
import sys 
from utils import Utils 
from BoardScene import BoardScene 
from View import View 

# app = QApplication(sys.argv)

In [ ]:
# # I cannot get Traces to adjust how I want them to. Start over. 
class PracticeTrace(QGraphicsLineItem): 
    arbitrary = QPointF(1e9, 1e9)
    
    def __init__(self, *args, traceWidth = 1, layers = ['F.Cu'], **kwargs): 
        super().__init__(*args, **kwargs)
        
        self.l1Anchors = dict()
        self.l2Anchors = dict() 
        self.l1s = dict() 
        self.l2s = dict()

        self.setLayers(layers)
        self.setTraceWidth(traceWidth)
        self.setPen(QPen(QColor(Qt.red) , 1))
        self.removed = False # Flag for mouseMoveEvent. Has this item been removed from scene 

    def getConnectedTraces(self): # Must pay attention to item.line().pn() vs self.line().pn() and p1 vs p2; this gets tricky
        connectedTraces = {'p1':[] , 'p2':[]}
        ids1 ,ids2 = [] , []
        print('SELF.LAYERS:', self.layers())
        for layer in self.layers(): 
            print('RTREE:', self.scene().rtrees[layer])

            ids1.extend( self.scene().rtrees[layer].intersection(self.p1SceneBounds()) ) # Hits at p1 
            ids2.extend( self.scene().rtrees[layer].intersection(self.p2SceneBounds()) ) # Hits at p2 

        print()
        print('IDS1:', ids1)
        print('IDS2:', ids2) 
        hitItems1 =  [ self.scene().ids[id] for id in ids1 if not (id is self.id()) ]
        hitItems2 =  [ self.scene().ids[id] for id in ids2 if not (id is self.id()) ]

        print('HITITEMS1:', hitItems1)
        print('HITITEMS2:', hitItems2)

        print('SELF.L0:', self.l0, self.l0.p1().toPoint() , self.l0.p2().toPoint())
        for hitItem in hitItems1: # Collect traces
            if isinstance(hitItem, Trace): 
                print('HITITEM:', hitItem, hitItem.line().p1().toPoint() , hitItem.line().p2().toPoint()) 
                if (hitItem.line().p1() == self.l0.p1()):
                    distal = hitItem.line().p2() # We may need to know connected_trace's distal point, to use as anchor
                    connectedTraces['p1'].append( ( hitItem , distal ) )
                elif ( hitItem.line().p2() == self.l0.p1()): 
                    distal = hitItem.line().p1()
                    connectedTraces['p1'].append( ( hitItem , distal ) )
                else: 
                    print('NEITHER p1 NOR p2 MATCHES UP' )
        for hitItem in hitItems2: 
            if isinstance(hitItem, Trace):
                if (hitItem.line().p1() == self.l0.p2()):
                    distal = hitItem.line().p2() # We may need to know connected_trace's distal point, to use as anchor
                    connectedTraces['p2'].append( ( hitItem , distal ) )
                elif ( hitItem.line().p2() == self.l0.p2()): 
                    distal = hitItem.line().p1()
                    connectedTraces['p2'].append( ( hitItem , distal ) )

        print('CONNECTEDTRACES:', connectedTraces)
        return connectedTraces
    
    def angleBetween(self, lineA, lineB): 
        # determines the angle between two lines 
        PI = math.pi
        
        alpha   = math.atan2( - lineA.dy() , lineA.dx() )
        beta    = math.atan2( - lineB.dy() , lineB.dx() ) 

        theta = self.normalizeAngle( beta - alpha )

        print('ANGLEBETWEEN l0 AND ln:', theta)
        if theta == math.pi/4: 
            print('ANGLE1 ACUTE') 
            return 'acute'
        elif theta == PI/2 or theta == 3*PI/2: 
            print('ANGLE1 PERPENDICULAR') 
            return 'perpendicular' 
        elif theta == 3*PI/4 or theta == 5*PI/4: 
            print('ANGLE1 OBTUSE') 
            return 'obtuse'
        else: 
            print('NSTH')
            
    def calculateAnchorAngleLine(self, connectedTraces, pn):  # Return 3-tuple ( anchor, angle, line ) 
        line = None 
        angle = 'acute' 
        


        if ( not connectedTraces) or ( len(connectedTraces) > 1 ):
            if  pn == 'p1': 
                anchor = self.l0.p1()                # Note we don't set line here, do it l8r bc acute angles actually require two lines, one line per side of l0. But we DO collect anchor here 
                
            elif pn == 'p2':
                anchor = self.l0.p2()
                
        elif len(connectedTraces) == 1: 
            item, distal = connectedTraces[0]
            anchor = distal 
            line = item.line() 
            angle = self.angleBetween(self.l0 , line )
            print('FOUND ANGLE TO BE :', angle)
            self.scene().removeItem(item)
            # Note if angle found to be acute we will l8r discard line in favor of two lines, one per side of l0. but if angle found to be obtuse or perpendicular, one line is all we need 

        if angle == 'acute': 
            if pn == 'p1':

                self.l1Anchors[ Utils.threePointOrientation(self.l0.p1() , self.l0.p2() , anchor) ] = anchor # TPO may be 0;inline, that is ok. May also be to a side; 1|2.  L8r, we will only .get( 1|2 , None ) to get any saved anchors to a side 
                print('SELF.L1ANCHORS:', self.l1Anchors)
                l1_1 = QLineF(self.l1Anchors.get(1, self.l0.p1()), self.arbitrary)
                l1_1.setAngle(self.l0.angle() - 45) 
                l1_2 = QLineF(self.l1Anchors.get(2, self.l0.p1()), self.arbitrary)
                l1_2.setAngle(self.l0.angle() + 45) 
                self.l1s[1] = l1_1 
                self.l1s[2] = l1_2 # Create two l1's , with angles dependent on seekerOrientation, and store them 

                print('SELF.L1S:', self.l1s)
        

            if pn == 'p2':
                self.l2Anchors[ Utils.threePointOrientation(self.l0.p1() , self.l0.p2() , anchor) ] = anchor
                
                l2_1 = QLineF(self.l2Anchors.get(1, self.l0.p2()), self.arbitrary) 
                l2_1.setAngle(self.l0.angle() - 135) 
                l2_2 = QLineF(self.l2Anchors.get(2, self.l0.p2()) , self.arbitrary) 
                l2_2.setAngle(self.l0.angle() + 135) 
                self.l2s[1] = l2_1 
                self.l2s[2] = l2_2 

        return anchor, angle, line
    
    def mousePressEvent(self, event): 
        self.l0 = self.line()
        self.t0 = self # Not a typo, self is Trace0
        self.l3 = None 
        self.moved = False # Flag if we did a mouseMoveEvent
        
        print('self.l0', self.l0)
        self.seekerOrientation = Utils.threePointOrientation(self.l0.p1() , self.l0.p2() , self.scene().seeker.scenePos()) # 0,1or2 representing seeker is inline with, or to a side of, l0
        print()
        print('SELF.SEEKERORIENTATION:', self.seekerOrientation)
        self.seekerSide = self.seekerOrientation # May initialize to 0 but never again 
        self.previousSeekerSide = self.seekerOrientation # save previous seeker orientation to know if changes 
        connectedTraces = self.getConnectedTraces()
        self.anchor1 , self.angle1, self.l1 = self.calculateAnchorAngleLine(connectedTraces['p1'] ,'p1') 
        print('SELF.ANCHOR1: ' , self.anchor1) 
        print('SELF.ANGLE1: ', self.angle1)
        print('SELF.L1:', self.l1)
        # self.initialAnchor1 = self.anchor1 

        self.anchor2 , self.angle2, self.l2 = self.calculateAnchorAngleLine(connectedTraces['p2'] , 'p2') 
        # self.initialAnchor2 = self.anchor2 

        self.li1 = QGraphicsLineItem() # Create/addToScene a standin QGLI representing our adjusted trace. QGLIs do not go into the scene.rtree. Add TraceItems upon mouseReleaseEvent; when user indicates they are done editing a trace.
        self.li1.setPen(QPen(Qt.red, self.traceWidth() , c = Qt.PenCapStyle.RoundCap)) 
        self.scene().addItem(self.li1) 
            
        self.li2=QGraphicsLineItem()
        self.li2.setPen(QPen(Qt.green, self.traceWidth(), c = Qt.PenCapStyle.RoundCap ))
        self.scene().addItem(self.li2)

        self.li3 = QGraphicsLineItem()
        self.li3.setPen(QPen(Qt.blue , self.traceWidth() , c = Qt.PenCapStyle.RoundCap))
        self.scene().addItem(self.li3)
                    
    def mouseMoveEvent(self, event): 
        self.moved = True 
        self.li3.show() # May be hidden later 
        # self.t0.hide() # hide self, l8r will be removed from scene in mre 
        print()
        
        if not self.seekerSide: # If seekerSide is still 0, then we have not moved to a side, and we can return 
            print('SEEKER NOT YET MOVED TO EITHER SIDE ')
            return 
        else: # if we have a seeker side, we want to...  
            print('SEEKER IS ON A SIDE ')
            
        self.seekerOrientation = Utils.threePointOrientation(self.l0.p1() , self.l0.p2() , self.scene().seeker.scenePos())
        print('SEEKERORIENTATION:', self.seekerOrientation)
        
        if self.seekerOrientation: # Set the seeker side to 1 or 2 but not 0
            self.seekerSide = self.seekerOrientation
            print('SEEKERSIDE:', self.seekerSide)
            
        if self.seekerOrientation == 0: # if seeker is inline w/ l0: 
            self.li3.setLine(QLineF(self.l0.p1() , self.l0.p2())) # Make li3 into origional lineItem
        
        # if self.seekerSide != self.previousSeekerSide: # then switched sides 
        #     print('SWITCHED SIDES')
        if self.angle1 == 'acute': 
            print('L1ANCHORS:', self.l1Anchors)
            print('L1S:', self.l1s)
            self.anchor1 = self.l1Anchors.get(self.seekerSide , self.l0.p1()) # Get the anchor or set it to p1 if no anchor recorded 
            self.l1 = self.l1s[self.seekerSide]
            

        if self.angle2 == 'acute': 
            print('L2ANCHORS:', self.l2Anchors)
            print('L2S:', self.l2s)
            self.anchor2 = self.l2Anchors.get(self.seekerSide , self.l0.p2()) 
            self.l2 = self.l2s[self.seekerSide]

        self.l3 = self.lineOffsetThroughPoint(self.l0 , self.scene().seeker.scenePos())
        
        isectL1L3 = self.intersection(self.l1, self.l3)  #NONETYPE l1 
        print('isectL1L3:', isectL1L3.toPoint())
        self.li1.setLine(QLineF(self.anchor1, isectL1L3)) # This may set                     


        isectL2L3 = self.intersection(self.l2, self.l3) 
        self.li2.setLine(QLineF(self.anchor2 , isectL2L3))
        
        self.li3.setLine( QLineF( self.li1.line().p2() , self.li2.line().p2() ))
        
        if self.seekerOrientation: 
            self.previousSeekerSide = self.seekerOrientation # Update previous seeker side if we were on a side; don't update it if we were at 0;inline.

        if ( not self.li1.line().isNull() ) and ( not self.li2.line().isNull() ): 
            segmentIntersectionLi1Li2 = self.segmentIntersection(self.li1.line(), self.li2.line()) # Check if l1/l2 SEGMENTS intersect, this can happen if both are acute and they 'overshoot' each other. We do NOT want to see this 'overshoot', so clip them at their intersection, if they do intersect. Also note that we don't care if their lines of infinite length intersect, just their segments. 
            if segmentIntersectionLi1Li2: 
                print('SEGMENT1and2 INTERSECT:', segmentIntersectionLi1Li2)
                # self.li1.line().setP2(segmentIntersectionLi1Li2) LineItems cannot modify their lines, they must have a new line set. So this won't do anything. Fails silently
                # self.li2.line().setP2(segmentIntersectionLi1Li2)
                self.li1.setLine(QLineF(self.li1.line().p1() , segmentIntersectionLi1Li2)) # Unfortunately, just to set P2, we have to set a whole new line bc QGLIs can't mutate their line. 
                self.li2.setLine(QLineF(self.li2.line().p1() , segmentIntersectionLi1Li2))
                self.li3.hide() 

    def mouseReleaseEvent(self, event):
        print('TRACE.RELEASEEVENT')
        super().mouseReleaseEvent(event) # call MyGraphicsObject.mRE to remove the trace, the trace we clicked on, self,  from the rtree, then put it BACK in the rtree, with its new position... which is useless... because we next .removeItem(self.t0)... but 
        
# add TraceItems to scene, if we were adjusting a trace, based on li123.
        # if self.adjusting: 
        #     self.adjusting = False

        if (self.l1 is not None) and (not self.l1.isNull()):
            t1 = PracticeTrace.fromPoints(self.layers() , self.traceWidth(), p1 = self.li1.line().p1() , p2 = self.li1.line().p2()) # not out of l1, but out of 1i1.line()
            # t1 = TraceItem(self.traceWidth(), self.layer(), self.li1.line()) # not out of l1, but out of 1i1.line()
            # t1.setPen(QPen(Qt.red, self.traceWidth() ,c = Qt.RoundCap))
            self.scene().addItem(t1)
            
        if (self.l2 is not None) and (not self.l2.isNull()):
            t2 = PracticeTrace.fromPoints(self.layers() , self.traceWidth(), self.li2.line().p1() , self.li2.line().p2())
            # t2 = TraceItem(self.traceWidth(), self.layer(), self.li2.line())

            # t2.setPen(QPen(Qt.green, self.traceWidth(), c= Qt.RoundCap))
            self.scene().addItem(t2)
            
        if (self.l3 is not None) and (not self.l3.isNull()) and (self.li3.isVisible()):
            t3 = PracticeTrace.fromPoints(self.layers(), self.traceWidth()  , self.li3.line().p1() , self.li3.line().p2())
            # t3 = TraceItem(self.traceWidth() , self. layer() , self.li3.line())
                
            # t3.setPen(QPen(Qt.blue, self.traceWidth(), c = Qt.RoundCap))
            self.scene().addItem(t3)
            
    
        self.scene().removeItem(self.li1)
        self.scene().removeItem(self.li2) # Li2 is None why
        self.scene().removeItem(self.li3)
        # self.scene().removeItem(self.test_item)
        
# Dont remove self.t0, if we only did a press-release
        if self.moved: 
            self.scene().removeItem(self.t0) # t0 is self, always removed after an adjust


    def p1SceneBounds(self):
        x1,y1= self.line().p1().toTuple()
        p1_bounds =  (x1,y1, x1,y1)
        return p1_bounds 
    
    def p2SceneBounds(self):
        x2,y2 = self.line().p2().toTuple()
        p2_bounds = (x2,y2, x2, y2)
        return p2_bounds 
    
    def layers(self):
        return self._layers
    def setLayers(self, layers ):
        self._layers = layers 

    def traceWidth(self): 
        return self._traceWidth 
    def setTraceWidth(self, traceWidth): 
        self.prepareGeometryChange() 
        self._traceWidth = traceWidth
        
    @staticmethod 
    def normalizeAngle(angle):
        while angle > 360: 
            angle-=360
        while angle < 360: 
            angle += 360 
        return angle 

    @staticmethod
    def intersection(line1, line2): # return intersection point of two QLine/Fs, or None if lines are  // parallel OR collinear OR one or both lines is of zero length. So QLineFs have a segment, defined by their two points, but this function will return intersection point where those lines intersect outside of their segment bounds. Compare against segments_intersect, which returns None if lines intersect outside their segments. 
        print('LINE1:', line1)
        print('LINE2:', line2)
        if line1.length() == 0 or line2.length() == 0: # Note distinction between l123 being of 0 length and li123 being of 0 length. l123 cannot be of 0 length, but li123 can be(and li are not used in this function)
            raise ValueError(f"line1.length(): {line1.length()} line2.length(): {line2.length()} but both lines must be of non-zero length to test intersection")

        intersection_type, intersection_point = line1.intersects(line2) # QLineF.intersects() is highly quirky
        
        if intersection_type is QLineF.IntersectionType.NoIntersection: 
            return None
        elif intersection_type is QLineF.IntersectionType.UnboundedIntersection:
            return intersection_point # Idc if the infinite length lines intersect, I only care if the segments intersect
        elif intersection_type is QLineF.IntersectionType.BoundedIntersection:
            return intersection_point

    @staticmethod
    def segmentIntersection(line1, line2): # Returns the intersection point, if segments intersect, else None. So if the infinitely long lines intersect outside of their segments, return None 
        if line1.length() == 0 or line2.length() == 0: 
            raise ValueError(f"line1.length(): {line1.length()} line2.length(): {line2.length()} but both lines must be of non-zero length to test intersection")
        
        intersection_type, intersection_point = line1.intersects(line2)
        if intersection_type == QLineF.IntersectionType.NoIntersection: 
            return None
        elif intersection_type == QLineF.IntersectionType.UnboundedIntersection:
            return None 
        elif intersection_type == QLineF.IntersectionType.BoundedIntersection:
            return intersection_point
    @staticmethod 
    def lineOffsetThroughPoint(line , point):
        offset = line.p1() - point
        p1 = line.p1() - offset
        p2 = line.p2() - offset 
        
        l = QLineF(p1 , p2) 
        return l
    
    @classmethod
    def fromPoints(cls, layers, traceWidth, p1, p2): 
        return cls( p1.x() , p1.y() , p2.x() , p2.y() , layers=layers, traceWidth=traceWidth, )

scene = BoardScene() 
t1 = PracticeTrace(30,50, 100,100) 
scene.addItem(t1)
view = View() 
view.setScene(scene) 

view.show() 


sys.exit(qApp.exec())




        

In [ ]:
# Example: How to add a Via onto scene using 'Add Via' button 
# Subclass QGraphicsScene
# Subclass QGraphicsItem 
# Reimplement mousePress, mouseMove, in bothm QGS and QGI
# Reimplement QGI.mouseRelease: Take on new net on mouseRelease,  if net is none and resolvedNet is not 'unresolved'

class Scene(QGraphicsScene): 
    def __init__(self):
        super().__init__()
        self.setMode(Utils.BoardSceneMode.NormalMode)

    def mousePressEvent(self, event): 
        if self.mode() == Utils.BoardSceneMode.NormalMode: 
            super().mousePressEvent(event)
        elif self.mode() == Utils.BoardSceneMode.AddViaMode: # In Scene.addViaModemousePressEvent, have the via take on nets below if appropriate 
            if ( self.via.net() == None ) and (self.via.resolvedNet != 'unresolved'): # None nets take on other nets upon mouseRelease
                self.via.setNet(self.via.resolvedNet)
            self.setMode(Utils.BoardSceneMode.NormalMode)

    def mouseMoveEvent(self, event):
        if self.mode() == Utils.BoardSceneMode.NormalMode:
            super().mouseMoveEvent(event)
        elif self.mode() == Utils.BoardSceneMode.AddViaMode:
            self.addViaModeMouseMoveEvent(event)

    def addViaModeMouseMoveEvent(self, event): 
        print('MOUSEMOVEEVENT')
        self.via.tentativeMove( Utils.snapToGrid(event.scenePos(), 20) )# MOve here, as long as no conflicts
            
    def mode(self):
        return self._mode 
    
    def setMode(self, mode): 
        self._mode = mode 
        if mode == Utils.BoardSceneMode.AddViaMode: 
            print('ENTERED ADD VIA MODE')
            self.via = PracticeVia(-20,-20,40,40)
            self.addItem(self.via) 
            self.via.setPos(-1e9,-1e9)
            self.views()[0].setMouseTracking(True) # mouseMoveEvent fires while no mouse button pressed down    
        elif mode == Utils.BoardSceneMode.NormalMode: 
            print('ENTERED NORMAL MODE')
            
class MainWindow(QMainWindow): 
    def __init__(self, *args , **kwargs): 
        super().__init__(*args, **kwargs) 
        self.addViaAction = QAction('Add Via' ,self, triggered = self.onAddViaActionTriggered)
        self._toolbar = self.addToolBar('Toolbar1')
        self._toolbar.addAction(self.addViaAction)
        self._toolbar.setAllowedAreas(Qt.ToolBarArea.AllToolBarAreas)
        

    def onAddViaActionTriggered(self): 
        self.centralWidget().scene().setMode(Utils.BoardSceneMode.AddViaMode)
        
class PracticeVia(QGraphicsEllipseItem): 

    def __init__(self, *args, net=None,  **kwargs):
        super().__init__(*args, **kwargs)
        self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsMovable | QGraphicsItem.GraphicsItemFlag.ItemIsSelectable)
        self.setBrush(Qt.darkCyan)

        self.setNet(net)

    def mouseReleaseEvent(self, event):
        if ( self.net() == None ) and (self.resolvedNet != 'unresolved'): # None nets take on other nets upon mouseRelease
            self.setNet(self.resolvedNet)
        super().mouseReleaseEvent(event) 
        
    def mousePressEvent(self, event): 
        self._offset = event.scenePos() - self.scenePos()
        
    def mouseMoveEvent(self, event): 
        pos = Utils.snapToGrid(event.scenePos() - self._offset , 20)
        self.tentativeMove(pos) 

    def tentativeMove(self, pos): # Move here but move back if there are obstructions 
        self._previousPos = self.scenePos()  # Save the previous position 
        # self._previousNet = self.net()          # Save the previous net 
        self.setPos(pos) # Move to proposed position
        nets = self.netsBeneath()  # Accumulate list of all nets beneath this item. # Are there net conflicts, if so, we do NOT want to move here. Did a None net collide with another net? If so, None net joins to other net 
        self.resolvedNet = self.resolveNets(nets) # resolve nets into one net if possible, ex 'GND' or None. Else set net 'unresolved'
        # self.setNet(resolve) 
        print('RESOLVEDNET:', self.resolvedNet)
        
        print('SELF.NET:', self.net())
        if (self.resolvedNet == 'unresolved'):
            self.setPos(self._previousPos) 
            # self.setNet(self._previousNet) 
        
        # elif ( (self._previousNet is not None) and (self._previousNet != self.resolvedNet ) ) : # if moving here is disallowed, reset to the previous position, AND previous net
        elif ( (self.net() is not None) and (self.net() != self.resolvedNet ) ) : # if moving here is disallowed, reset to the previous position, AND previous net
            self.setPos(self._previousPos) 
            # self.setNet(self._previousNet) 

    def netsBeneath(self): # Return list of all nets beneath this item 
        netsBeneath = set([self.net()])
        for item in self.scene().items(): 
            if self.collidesWithItem(item): 
                netsBeneath.add(item.net())
        return netsBeneath
    
    def resolveNets(self, nets): # Return True if a net is resolvable from given list of nets 
        nonNoneNets = [net for net in nets if net != None] 
        
        if len(nonNoneNets) == 0: # Then net was None, which is allowed
            return None 
              
        elif len(nonNoneNets) == 1: 
            if (self.net() is not None) and (self.net() != nonNoneNets[0]): # 
                return 'unresolved'
            else: 
                return nonNoneNets[0] 
            
        elif len(nonNoneNets) >1 : 
            return 'unresolved'

    def net(self):
        return self._net 
    def setNet(self, net):
        self._net = net


v1 = PracticeVia(-20,-20,40,40, net = 'GND')
v1.setPos(100,100)
v2 = PracticeVia(-20,-20,40,40, net = '3V3')
v2.setBrush(Qt.red)

scene = Scene()
view = QGraphicsView() 
view.setScene(scene)

mw = MainWindow()
mw.setCentralWidget(view)
mw.show() 

scene.addItem(v1)
scene.addItem(v2)

sys.exit(qApp.exec())
        
    

In [ ]:
# Example of how to reimplement mouseMoveEvent to NOT move to disallowed locations
# ToDo: Special behavior for None nets, which take on first net they're placed on 
def snapToGrid(pos): 
    gridSpacing = 10.0
    return QPointF(int(pos.x()/gridSpacing) *gridSpacing , int(pos.y()/gridSpacing)*gridSpacing)

class PracticeVia(QGraphicsEllipseItem): 

    def __init__(self, *args, net=None,  **kwargs):
        super().__init__(*args, **kwargs)
        self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsMovable | QGraphicsItem.GraphicsItemFlag.ItemIsSelectable)
        self.setBrush(Qt.darkCyan)

        self.setNet(net)
        
    def mousePressEvent(self, event): 
        self._offset = event.scenePos() - self.scenePos()
        
    def mouseMoveEvent(self, event): 
        self._previousPos = self.scenePos()  # Save the previous position 
        self._previousNet = self.net()          # Save the previous net 
        self.setPos(snapToGrid(event.scenePos() - self._offset)) # Move to proposed position
        self.resolveNets()  # Are there net conflicts, if so, we do NOT want to move here. Did a None net collide with another net? If so, None net joins to other net 
        
        print('SELF.NET:', self.net())
        if (self.net() == 'unresolved'):
            self.setPos(self._previousPos) 
            self.setNet(self._previousNet) 
        
        elif ( (self._previousNet is not None) and (self._previousNet != self.net() ) ) : # if moving here is disallowed, reset to the previous position, AND previous net
            self.setPos(self._previousPos) 
            self.setNet(self._previousNet) 

    def resolveNets(self): 
        nets = set([self.net()])
        for item in self.scene().items(): 
            if self.collidesWithItem(item): 
                nets.add(item.net())

        nonNoneNets = [net for net in nets if net != None] 
        
        if len(nonNoneNets) == 0: 
            self.setNet(None)
        elif len(nonNoneNets) == 1: 
            if (self._previousNet is not None) and (self._previousNet != nonNoneNets[0]): 
                self.setNet('unresolved')
            else: 
                self.setNet(nonNoneNets[0]) # item with None net takes on 
        elif len(nonNoneNets) >1 : 
            self.setNet('unresolved')


    def net(self):
        return self._net 
    def setNet(self, net):
        self._net = net


v1 = PracticeVia(-20,-20,40,40, net = 'GND')
v1.setPos(100,100)
v2 = PracticeVia(-20,-20,40,40, net = '3V3')
v2.setBrush(Qt.red)

scene = QGraphicsScene()
view = QGraphicsView() 
view.setScene(scene)
view.show()

scene.addItem(v1)
scene.addItem(v2)

sys.exit(app.exec())
        
    

In [ ]:
str(None)

In [ ]:
QGraphicsItem.GraphicsItemFlag.ItemSendsScenePositionChanges
vs
QGraphicsItem.GraphicsItemFlag.ItemSendsGeometryChanges : This flag enables sending of ItemPositionChange & ItemPositionHasChanged & other position & transform changes. 


note QGraphicsItem.setFlags() rewrites all flags every time you call it, must include all flags you wanna set in one go





In [ ]:
12/5

In [ ]:
# from qgraphicsitem.cpp: 

QRectF QGraphicsLineItem::boundingRect() const
{
    Q_D(const QGraphicsLineItem);
    if (d->pen.widthF() == 0.0) {
        const qreal x1 = d->line.p1().x();
        const qreal x2 = d->line.p2().x();
        const qreal y1 = d->line.p1().y();
        const qreal y2 = d->line.p2().y();
        qreal lx = qMin(x1, x2);
        qreal rx = qMax(x1, x2);
        qreal ty = qMin(y1, y2);
        qreal by = qMax(y1, y2);
        return QRectF(lx, ty, rx - lx, by - ty);
    }
    return shape().controlPointRect();
}



In [ ]:
Trace adjust is not precise. Needs new strategy. (slope function had float inaccuracy, plus switched from 1e3 to 1e9 ) Ex trace laid at multiple of 45 degrees. but after dragging , so slightly off of 45n degrees. Really bad. Also it don't work for perp connections.
Traces must avoid different nets 


In [ ]:
# I cannot figure out how to get traces to move correctly. Changing to some examples for practice 
# https://doc.qt.io/qtforpython-6/tutorials/datavisualize/index.html#datavisualize-index


In [ ]:
# Ch1 - Reading data from a csv 

import os 
import pandas as pd 
from PySide6.QtCore import QDateTime, QTimeZone 

filePath = os.path.join('C:\\','Users' , 'robby', 'Coding', 'datasets', 'earthquakes', 'all_day.csv')

def transformDate(utc, timezone = None): 
    utcFormat = "yyyy-MM-ddTHH:mm:ss.zzzZ"
    newDate = QDateTime().fromString(utc, utcFormat) 
    if timezone: 
        newDate.setTimeZone(timezone)
    return newDate

def readData(filePath): 
    df = pd.read_csv(filePath) 

    df = df.drop(df[df.mag<0].index)
    magnitudes = df['mag']

    timezone = QTimeZone(b'America/New_York')

    times = df['time'].apply(lambda x: transformDate(x , timezone))

    return times, magnitudes 

data = readData(filePath)
# data = list(zip(data[0] , data[1]))
# print(data)
print(data[0][0] , data[1][0]) 




In [ ]:
from PySide6.QtWidgets import QApplication
import sys
app = QApplication(sys.argv)

In [ ]:
from PySide6.QtWidgets import QMainWindow
from PySide6.QtGui import QKeySequence, QIcon

class MainWindow(QMainWindow): 
    def __init__(self, centralWidget):
        super().__init__()
        self.setWindowTitle('Earthquake Info') 
        self.setCentralWidget(centralWidget)
        fileMenu = self.menuBar().addMenu('File')
        fileMenu.addAction(QIcon.fromTheme(QIcon.ThemeIcon.ApplicationExit), 
                            'Exit',
                            QKeySequence.StandardKey.Quit , 
                            self.close)
        self.statusBar().showMessage('Data loaded and plotted')

        geometry = self.screen().availableGeometry() 
        self.setFixedSize(geometry.width() * .8 , geometry.height() *.7) 
# Ch4 - Add a QTableView 
# We would use the default item model that comes with a QTableWidget. This can reduce your codebase, as you don't need to implement a data model, however it 'cannot be used with just any data' (like what?) 
# We will implement a custom model, allowing us to set headers, manipulate the formats of the UTC/float data, set style properties like text alignment, and color properties for the cell/contents 

from PySide6.QtCore import Qt, QAbstractTableModel, QModelIndex
from PySide6.QtGui import QColor 

class TableModel(QAbstractTableModel):
    def __init__(self, data=None):
        super().__init__()
        self.loadData(data)

    def loadData(self, data): 
        self.inputDates = data[0].values
        self.inputMagnitudes = data[1].values 
        self._columnCount = 2 
        self._rowCount = len(self.inputMagnitudes)

    def rowCount(self, parent = QModelIndex()): 
        return self._rowCount

    def columnCount(self, parent = QModelIndex()): 
        return self._columnCount 

    def headerData(self, section, orientation, role):
        if role != Qt.ItemDataRole.DisplayRole: 
            return None 
        if orientation == Qt.Orientation.Horizontal: 
            return ( "Date", "Magnitude")[section]
        else: 
            return f'{section}'

    def data(self, index, role = Qt.ItemDataRole.DisplayRole): 
        column = index.column() 
        row = index.row() 

        if role == Qt.ItemDataRole.DisplayRole: 
            if column == 0 : 
                date = self.inputDates[row].toPython()
                return str(date)[:3]

            elif column == 1: 
                magnitude = self.inputMagnitudes[row] 
                return f'{magnitude:.2f}'

        elif role == Qt.ItemDataRole.BackgroundRole: 
            return QColor(Qt.GlobalColor.white)

        elif role == Qt.ItemDataRole.TextAlignmentRole: 
            return Qt.AlignmentFlag.AlignRight 

        return None 


# Create a widget that has a QTableView, and connect it to TableModel 
# Ch6: plot the csv data by converting it to a QLineSeries. Modify the axis to properly display QDateTime on the xaxis, and magnitude values on the yaxis 
from PySide6.QtWidgets import QHBoxLayout, QHeaderView, QSizePolicy, QTableView, QWidget
from PySide6.QtQuickWidgets import QQuickWidget
from PySide6.QtGraphs import QLineSeries, QDateTimeAxis, QValueAxis, QGraphsTheme
import math 

class Widget(QWidget): 
    def __init__(self, data=None): 
        super().__init__() 

        self.model = TableModel(data)

        self.tableView = QTableView() 
        self.tableView.setModel(self.model)

        self.tableView.horizontalHeader().setSectionResizeMode(QHeaderView.ResizeMode.ResizeToContents)
        self.tableView.verticalHeader().setSectionResizeMode(QHeaderView.ResizeMode.ResizeToContents)
        self.tableView.horizontalHeader().setStretchLastSection(True)

        # Create QGraphView via QML
        self.populateSeries()
        self.quickWidget = QQuickWidget(self) 
        self.quickWidget.setResizeMode(QQuickWidget.ResizeMode.SizeRootObjectToView)
        self.theme = QGraphsTheme()
        self.theme.setTheme(QGraphsTheme.Theme.BlueSeries)
        initialProperties = {'theme': self.theme , 
                             'axisX': self.axisX , 
                             'axisY': self.axisY ,
                             'seriesList': self.series}
        
        self.quickWidget.setInitialProperties(initialProperties) 
        self.quickWidget.loadFromModule('QtGraphs' , 'GraphsView')

        self.setLayout(QHBoxLayout(self))
        size = QSizePolicy( QSizePolicy.Policy.Preferred , QSizePolicy.Policy.Preferred)


        size.setHorizontalStretch(1)
        self.tableView.setSizePolicy(size)
        self.layout().addWidget(self.tableView)

        size.setHorizontalStretch(4) 
        self.quickWidget.setSizePolicy(size)
        self.layout().addWidget(self.quickWidget)


    def populateSeries(self): 
        def seconds(qtime):
            return qtime.minute() * 60 + qtime.second()
        
        self.series = QLineSeries() 
        self.series.setName('Magnitude (Column 1)')
        
        timeMin = QDateTime(2100 , 1, 1, 0,0,0) 
        timeMax = QDateTime(1970 ,1 ,1 ,0,0,0) 
        timeZone = QTimeZone(QTimeZone.Initialization.UTC)
        yMin = 1e37 
        yMax = -1e37
        dateFormat = 'yyyy-MM-dd HH:mm:ss.zzz'
        for i in range(self.model.rowCount()):
            t = self.model.index(i,0).data()
            time = QDateTime.fromString(t, dateFormat)
            time.setTimeZone(timeZone) 
            y = float(self.model.index(i, 1).data())
            if time.isValid() and y>0:
                if time > timeMax: 
                    timeMax = time 
                if time < timeMin: 
                    timeMin = time 
                if y > yMax: 
                    yMax = y 
                if y < yMin: 
                    yMin = y 
                self.series.append(time.toMSecsSinceEpoch() , y)


        self.axisX = QDateTimeAxis() 
        self.axisX.setLabelFormat('dd.MM (h:mm)')
        self.axisX.setTitleText('Date')
        self.axisX.setMin(timeMin.addSecs(-seconds(timeMin.time())))
        self.axisX.setMax(timeMax.addSecs(3600 - seconds(timeMax.time())))
        self.series.setAxisX(self.axisX)
        
        self.axisY = QValueAxis()
        self.axisY.setLabelFormat('%2f')
        self.axisY.setTitleText('Magnitude')
        self.axisY.setMin(math.floor(yMin))
        self.axisY.setMax(math.ceil(yMax))
        self.series.setAxisY(self.axisY)





                
            
        
# widget = Widget(data) 
# mw = MainWindow(widget) # Nothing happens. This should be showing the graph

mw = MainWindow(QWidget()) # Something happens( blank MainWindow )

mw.show()
sys.exit(app.exec())

# Nothing happens 

In [ ]:
Coordinated Universal Time UTC: primary global standard used to regulate time. 

In [ ]:
import math
math.atan2(10, 10) * 180/math.pi

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
# print(QLineF().angle())

if QLineF().isNull(): 
    print('null')
if QLineF(0,0,0,0).isNull(): 
    print('null')
if QLineF(10,10,10,10).isNull(): 
    print('null')
    

from utils import Utils 
p1, p2, p3 = QPoint(0,0) , QPoint(10,0) , QPoint(0,10) #  This should be cw aka 1
ori = Utils.threePointOrientation(p1,p2,p3)
print('Ori:', ori) # ccw all checks out , but the QT Coordinate system is flipped in y 

p4,p5,p6 = QPoint(0,0) , QPoint(10,0) , QPoint(0 , -10) # This should be ccw aka 2 
ori = Utils.threePointOrientation(p4,p5,p6)
print('Ori2:', ori)




In [ ]:
https://stackoverflow.com/questions/13102787/prevent-qgraphicsitem-from-moving-outside-of-qgraphicsscene
This SO comment makes a case for why youd use .itemChange rather than mouseMOveEvent: Ex in multiply selected items, only one item receives a mouseMoveEvent. ElasticNodes example uses .itemChange for much the same reason, items are moving without mouseMoveEvents

In [ ]:
print(0 is 0)

In [ ]:
# TraceBase, TraceItem, and Trace, from QGraphicsItem

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
import sys
from utils import app, Utils, CopperItemContainer
from BoardScene import BoardScene
from BoardView import BoardView
from MainWindow import MainWindow

class TraceBase(): 
    def __init__(self, x1, y1, x2, y2, traceWidth, *args, **kwargs ): 
        super().__init__(*args, **kwargs)
        self._x1 = x1 
        self._y1 = y1
        self._x2 = x2
        self._y2 = y2 
        self._traceWidth = traceWidth

        # self.setPen(QPen(Qt.black, traceWidth , c = Qt.PenCapStyle.RoundCap))
        # self.setBrush(QBrush(Qt.black))

    def boundingRect(self): 
        width = self.x2()- self.x1()
        height= self.y2() - self.y1() 
        
        return QRectF( *self.p1() , width, height).normalized().adjusted(-self._traceWidth/2 , -self._traceWidth/2 , self._traceWidth/2 , self._traceWidth/2)

        
    def shape(self):
        path = QPainterPath()
        path.moveTo(QPointF(*self.p1()))
        path.lineTo(QPointF(*self.p2()))
        stroker = QPainterPathStroker() # In computer graphics, 'stroking' is the known difficult problem of offsetting shapes. Qt uses it to calculate 'fillable outlines of shapes': Give a path, get an offset of that path. note it strokes the 'inside' and 'outside' of the given shape, so there's two
        stroker.setWidth(self.traceWidth()) 
        stroker.setJoinStyle(Qt.RoundJoin)
        stroker.setCapStyle(Qt.RoundCap)        
        path = stroker.createStroke(path)
        return path 
    
    def traceWidth(self): 
        return self._traceWidth 
        
    def pen(self):
        return self._pen
    
    # def brush(self): QGLI has no brush. Trace should have no brush as well 
    #     return self._brush 

    def x1(self) :
        return self._x1 
    def y1(self): 
        return self._y1 
    def x2(self): 
        return self._x2 
    def y2(self): 
        return self._y2
    
    def p1(self):
        return (self._x1, self._y1)
    def setP1(self, p1 ): 
        self.prepareGeometryChange()
        self._x1 , self._y1 = p1 
        
    def p2(self):
        return (self._x2 , self._y2)
    def setP2(self, p2): 
        self.prepareGeometryChange()
        self._x2 , self._y2 = p2

    def line(self):
        return (self.p1() , self.p2() )    
    def setLine(self, x1=None, y1=None, x2=None , y2=None , p1=None , p2=None , line=None ): 
        self.prepareGeometryChange()
        
        if x1 and y1 and x2 and y2: 
            self._x1 = x1 
            self._y1 = y1 
            self._x2 = x2
            self._y2 = y2
            
        elif p1 and p2: 
            self._x1 = p1[0]
            self._y1 = p1[1] 
            self._x2 = p2[0] 
            self._y2 = p2[1] 
            
        elif line: 
            if len(line)==4: 
                self._x1 = line[0]
                self._y1 = line[1]
                self._x2 = line[2] 
                self._y2 = line[3]

class TraceItem(TraceBase, QGraphicsItem): 
    def __init__(self, x1, y1, x2, y2, layer, traceWidth, parent, *args, **kwargs ): 
        super().__init__(x1, y1, x2, y2, traceWidth, parent=parent, *args, **kwargs)

        self._layer = layer

        self.setPen(QPen(Utils.layerColors[layer], traceWidth , c = Qt.PenCapStyle.RoundCap))

        # self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsSelectable | QGraphicsItem.ItemIsMovable)
            
    def paint(self, painter, option, widget): 
        painter.setPen(self.pen())
        painter.drawLine(self.x1() , self.y1() , self.x2() , self.y2())
    
    def setTraceWidth(self, traceWidth): 
        self.prepareGeometryChange() 
        self._traceWidth = traceWidth
        self._pen = QPen(self.pen().color() , traceWidth, c = Qt.PenCapStyle.RoundCap)
        
    def pen(self):
        return self._pen
    def setPen(self, pen): 
        self.prepareGeometryChange() 
        self._traceWidth = pen.width() 
        self._pen = QPen(Utils.layerColors[self._layer] , pen.width(), c = Qt.PenCapStyle.RoundCap)



class Trace(TraceBase, CopperItemContainer, QGraphicsItem): 
    def __init__(self, x1, y1, x2, y2, layers, traceWidth, parent=None, *args, **kwargs ): 
        super().__init__(x1=x1, y1=y1, x2=x2, y2=y2, layers = layers, traceWidth=traceWidth, parent=parent, *args, **kwargs)
        self._x1 = x1 
        self._y1 = y1
        self._x2 = x2
        self._y2 = y2 
        self._traceWidth = traceWidth

        self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsSelectable | QGraphicsItem.ItemIsMovable)

        for layer in self.layers(): 
            traceItem = TraceItem(self._x1 , self._y1 , self._x2 ,self._y2 , layer, traceWidth, self) 
            
    def paint(self, painter, option ,widget): 
        pass 

    def setLine(self, line): 
        super().setLine(line) # TraceBase.setLine
        for child in self.childItems(): #Control child TraceItem lines
            if isinstance(child, TraceItem): 
                child.setLine(line) 
                
    def setTraceWidth(self, traceWidth): 
        self.prepareGeometryChange() 
        self._traceWidth = traceWidth
        self._pen = QPen(self.pen().color() , traceWidth, c = Qt.PenCapStyle.RoundCap)
        
        for child in self.childItems(): # Control child TraceItems 
            if isinstance(child, TraceItem): 
                child.setTraceWidth(traceWidth)

    def sceneTerminals(self):
        return self._sceneTerminals
    def setSceneTerminals(self):
        self._sceneTerminals = [self.p1() , self.p2()]





scene = BoardScene()
view = BoardView() 

originMarker = QGraphicsEllipseItem(-10,-10,20,20) 
originMarker.setPen(QPen(Qt.magenta , 1))
scene.addItem(originMarker)

# trace = TraceItem(30,50, 100,100 ,'F.Cu')
traceWidth = 1 
trace = Trace(30,50 ,100,100, ['B.Cu', 'F.Cu'] , traceWidth)
# trace.setLine(100,100, 200,200)
scene.addItem(trace)

view.setScene(scene) 
view.show() 
sys.exit(app.exec())

        

In [ ]:
# Here's why we cannot(reasonably) subclass QGraphicsLineItem for use in Trace: It cannot handle **kwargs input 
# This is also why TraceBase does not inherit QGraphicsItem; why Trace&TraceItem each inherit QGraphicsItem directly
# unfortunately you DO have to master cooperative inheritance/multiple inheritance to really understand this :(
from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *

# class Trace(QGraphicsLineItem):
#     def __init__(self, *args, **kwargs): 
#         super().__init__(*args, **kwargs)

# class A(Trace): 
#     def __init__(self, a, *args, **kwargs):
#         print('A.ARGS:', args)
#         print('A.KWARGS:', kwargs)
#         super().__init__(*args, **kwargs) 

# a = A('a',  line= QLineF())
# print(a.line())



# However, we need not forward kwargs to the constructor. We can super().__init__() without any arguements, then use setters post-constructor.
class Trace(QGraphicsLineItem):
    def __init__(self, **kwargs):
        super().__init__()  # Call parent with no args
        self.setLine(kwargs['line']) # Use setter method post-constructor. We need not forward kwargs to the constructor, which cannot handle kwarg forwarding


class A(Trace):
    def __init__(self, a, **kwargs):
        print('A.ARGS:', a)
        print('A.KWARGS:', kwargs)
        super().__init__(**kwargs)

# problem: argument is 'hiding' in kwargs. Terrible practice? Workaround : don't use QGLI, diy QGI. This is hard, however. need prepareGeometryChange, lots of getters/setters. 

a = A('a', line=QLineF(10,10,20,20))
print(a.line())

In [ ]:
# Create TraceItem from QGraphicsItem. All this work just to use *args/**kwargs? 

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
import sys
from utils import app

class TraceItem(QGraphicsItem): 
    def __init__(self, x1, y1, x2, y2, parent=None, *args, **kwargs ): 
        super().__init__(*args, **kwargs)
        self._x1 = x1 
        self._y1 = y1
        self._x2 = x2
        self._y2 = y2 


        self.setPen(QPen(Qt.black, 1 , c = Qt.PenCapStyle.RoundCap))
        self.setBrush(QBrush(Qt.black))

        self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsSelectable | QGraphicsItem.ItemIsMovable)
        

    def boundingRect(self): 
        width = self.x2()- self.x1()
        height= self.y2() - self.y1() 
        
        return QRectF( *self.p1() , width, height).normalized().adjusted(-self._traceWidth/2 , -self._traceWidth/2 , self._traceWidth/2 , self._traceWidth/2)
    
    def paint(self, painter, option, widget): 
        painter.setPen(self.pen())
        painter.setBrush(self.brush())
        painter.drawLine(self.x1() , self.y1() , self.x2() , self.y2())
        
    def shape(self):
        path = QPainterPath()
        path.moveTo(QPointF(*self.p1()))
        path.lineTo(QPointF(*self.p2()))
        stroker = QPainterPathStroker() # In computer graphics, 'stroking' is the known difficult problem of offsetting shapes. Qt uses it to calculate 'fillable outlines of shapes': Give a path, get an offset of that path. note it strokes the 'inside' and 'outside' of the given shape, so there's two
        stroker.setWidth(self.traceWidth()) 
        stroker.setJoinStyle(Qt.RoundJoin)
        stroker.setCapStyle(Qt.RoundCap)        
        path = stroker.createStroke(path)
        return path 
    
    def traceWidth(self): 
        return self._traceWidth 
    def setTraceWidth(self, traceWidth): 
        self.prepareGeometryChange() 
        self._traceWidth = traceWidth
        self._pen = QPen(self.pen().color() , traceWidth, c = Qt.PenCapStyle.RoundCap)
        
    def pen(self):
        return self._pen
    def setPen(self, pen): 
        self.prepareGeometryChange() 
        self._traceWidth = pen.width() 
        self._pen = QPen(pen.color() , pen.width(), c = Qt.PenCapStyle.RoundCap)

    def brush(self): 
        return self._brush 
    def setBrush(self, brush): 
        self._brush = brush

    def x1(self) :
        return self._x1 
    def y1(self): 
        return self._y1 
    def x2(self): 
        return self._x2 
    def y2(self): 
        return self._y2
    
    def p1(self):
        return (self._x1, self._y1)
    def setP1(self, p1 ): 
        self.prepareGeometryChange()
        self._x1 , self._y1 = p1 
        
    def p2(self):
        return (self._x2 , self._y2)
    def setP2(self, p2): 
        self.prepareGeometryChange()
        self._x2 , self._y2 = p2

    def line(self):
        return (self.p1() , self.p2() )    
    def setLine(self, x1=None, y1=None, x2=None , y2=None , p1=None , p2=None , line=None ): 
        self.prepareGeometryChange()
        
        if x1 and y1 and x2 and y2: 
            self._x1 = x1 
            self._y1 = y1 
            self._x2 = x2
            self._y2 = y2
            
        elif p1 and p2: 
            self._x1 = p1[0]
            self._y1 = p1[1] 
            self._x2 = p2[0] 
            self._y2 = p2[1] 
            
        elif line: 
            if len(line)==4: 
                self._x1 = line[0]
                self._y1 = line[1]
                self._x2 = line[2] 
                self._y2 = line[3]
                
scene = QGraphicsScene()
view = QGraphicsView() 

originMarker = QGraphicsEllipseItem(-10,-10,20,20) 
originMarker.setPen(QPen(Qt.magenta , 1))
scene.addItem(originMarker)

trace = TraceItem(30,50, 100,100)
# trace.setLine(100,100, 200,200)
trace.setPen(QPen(Qt.blue, 10 , c = Qt.PenCapStyle.RoundCap))
scene.addItem(trace)

view.setScene(scene) 
view.show() 
sys.exit(app.exec())

        

In [ ]:
l = [0,1,2,3]
print(l[0:2])
print(l[2:])

In [ ]:
# Reverifying Trace 

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
from LayersItem import *
import sys
from MainWindow import MainWindow

from Trace import Trace

window = MainWindow()
boardScene = window.centralWidget().widget(1).scene()

line = QGraphicsLineItem(QLineF( 50,50 , 1000, 400))
trace = Trace( layers= ['F.Cu'] , traceWidth = 1 , p1 = QPointF(40,50 ) , p2 = QPointF(1000, 400) )
boardScene.addItem(trace)
boardScene.addItem(line)

window.show()
sys.exit(qApp.exec())

In [ ]:
# My Viewport dots are only appearing in one quadrant; xy positive. How do I get them in all quadrants? 
# A: offset by the exposed rect : painter.drawPoint( x*tickSpacing + rect.left()) , y*tickSpacing + rect.top()) 
# Great, but now there is a 'flowing' effect, appears as if dots flow behind items when zooming. (This because dots start drawing at rect leftTop. 
# A: Start drawing dots on grid, at grid pos nearest leftTop





In [ ]:
# My viewport is constrained to scroll only in a certain area. I want to be able to scroll on an effectively infinte area
# Answer: Set a large scroll area with :
#     view.setSceneRect(-10000, -10000, 20000, 20000)

# ### QGraphicsView ### 
# Visualize the contents of a QGraphicsScene in a scrollable viewport. 

# Scroll to any position on the scene using the scrollbars,(umm yet I cannot)(What is my sceen rectangle? unset, so infinite) or by calling .centerOn(QPoint)
# The visualized area is by default detected with QGS.itemsBoundingRect(), which returns the bounding rect of all items on the scene. 
# Use QGVIEW.setSceneRect() to set the visualized area yourself. This will adjust the scroll bars' ranges. 

# QGraphicsView.setBackground(painter, rect) default fills rect using the view's background brush. If no such brush defined(the default), the scene's .drawBackground is called instead. 

In [ ]:
# Set Background to Draw Dots Sensibly
from utils import * 
from View import View
from SchematicScene import SchematicScene

class GridView(View ): 


    def drawBackground(self, painter, rect): 

        print()
        print('DRAWBACKGROUND')

        painter.setBrush(Qt.black)
        painter.setPen(QPen(Qt.black, 1)) # Note QPen width 1 makes dots much more visible than width 0 
        # painter.setPen(Qt.NoPen) Makes points disappear
        


        # Note WHen zooming wayin to wayout , the PEN WIDTH of your painted points becomes important so the user can see it. 
        # I'm content with this for prototype, but production app should dynamically set pen widths based on zoom (?)
        def calculateTickSpacing():
            xScale = painter.transform().m11() # xScale is represented at the transformationMatrix m11 element. 
            print('XSCALE:', xScale)
            tickSpacing = Utils.boardTickSpacing
            if (xScale <.5): # ZOOMEDWAYOUT
                painter.setPen(QPen(Qt.black, 10)) # Set a wide pen so user can still see the dots 
                tickSpacing = Utils.boardTickSpacing*10
            elif .5 <= xScale <= 10: 
                painter.setPen(QPen(Qt.black, 1))
                tickSpacing = Utils.boardTickSpacing
            if xScale > 20: #ZOOMEDWAYIN
                painter.setPen(QPen(Qt.black, .1)) # set a thin pen so user can still see the dots 
                tickSpacing = Utils.boardTickSpacing/10

            return tickSpacing

        tickSpacing = calculateTickSpacing()
        print('TICKSPACING:', tickSpacing)
        
        numTicksX = int(rect.width()/tickSpacing) 
        numTicksY = int(rect.height()/tickSpacing) 

        xStart = int(rect.left() / tickSpacing) * tickSpacing # important to start drawing points snapped to grid. If start drawing points at rect.left()&rect.top(), induces a 'flowing' effect while zooming
        yStart = int(rect.top() / tickSpacing) * tickSpacing
        
        for i in range(numTicksX): 
            for j in range(numTicksY):
                x = i*tickSpacing + xStart 
                y =  j* tickSpacing + yStart
                painter.drawPoint(QPointF(x , y)) # Note pass a QPointF() to be able to use floats with .drawPoint() 
                # painter.drawEllipse(QPointF(x,y), 1, 1)

    def wheelEvent(self, event): # Wheel as in mouseWheel 
        
        delta = event.angleDelta().y() # How much mouseWheel scrolled
        scaleFactor = math.pow(2.0, -delta / 500)
        self.scaleScene(scaleFactor)

    def scaleScene(self, scaleFactor):
        zoom = self.transform().scale(scaleFactor, scaleFactor).m11() # Scale current transform to predict zoom. The x scale lives in the matrix's m11 element      #  Used to do this , which also works: .mapRect(QRectF(0, 0, 1, 1)).width() # QTransform.mapRect(rect) -> QRectF, mapped onto the given QTransform. Note that we gave a unit rectangle; a rectangle where width&height=1. So, we are testing to see how much a unit scales under this transform. Note that self.transform() includes any previous scaling; representing the currently applied zoom, which we should limit to a certain range 

        if zoom < 0.01 or zoom > 100: # Prevent crazy scale changes.
            return

        self.scale(scaleFactor, scaleFactor) 



        
view = GridView() 
scene = SchematicScene()
view.setScene(scene)
r = QGraphicsRectItem(-100,-100, 200,200) 
r.setBrush(Qt.blue) 
r.setPen(Qt.NoPen)
scene.addItem(r)

view.show()

sys.exit(app.exec())






In [ ]:
print(int(3.9))
print(round(3.9))

In [ ]:
# Workflow porting NetSym from Kicad: 3V3 symbol
from utils import * 
from kicadSymbolConverter import KicadSymbolConverter
# from kicadFootprintConverter import KicadFootprintConverter

officialKicadSymbolsLibrariesPath = os.path.join('third_party', 'kicad', 'symbols', 'kicad-symbols')

threeV3FilePath = os.path.join(officialKicadSymbolsLibrariesPath , 'power.kicad_symdir', '+3V3.kicad_sym')
threeV3SymFile = KicadSymbolConverter.convert(threeV3FilePath, categories = ['netSymbols']) # places '+3V3.sym' into the 'netSymbols' folder.Note 'categories so named bc supposed to be like ['capacitors', 'ceramic_capacitors']




